# California Housing Price Prediction
## A Professional Machine Learning Pipeline (LightGBM, Scikit-Learn)

This notebook demonstrates an end-to-end machine learning project to predict median house prices in California. It includes:
1. Exploratory Data Analysis (EDA)
2. Data Preprocessing & Feature Engineering
3. Model Training (Linear Regression, Random Forest, LightGBM)
4. Hyperparameter Tuning
5. Model Evaluation and Residual Analysis
6. Feature Importance Visualization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV

# Set plotting style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)

### 1. Data Loading & Initial Exploration

In [ ]:
housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target

print("Dataset Shape:", df.shape)
df.head()

In [ ]:
df.describe()

### 2. Exploratory Data Analysis (EDA)

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Feature Correlation Matrix")
plt.show()

In [ ]:
# Distribution of Target Variable
sns.histplot(df['MedHouseVal'], kde=True, color='green')
plt.title("Distribution of Median House Value (Target)")
plt.show()

### 3. Data Preprocessing & Outlier Removal

In [ ]:
# Handling Outliers for MedHouseVal (simple IQR method)
Q1 = df['MedHouseVal'].quantile(0.25)
Q3 = df['MedHouseVal'].quantile(0.75)
IQR = Q3 - Q1
df_clean = df[(df['MedHouseVal'] >= Q1 - 1.5*IQR) & (df['MedHouseVal'] <= Q3 + 1.5*IQR)]

X = df_clean.drop('MedHouseVal', axis=1)
y = df_clean['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set size: {X_train.shape}")

### 4. Pipeline & Modeling

In [ ]:
# Standard Scaling + LightGBM Tuned (GridSearch results applied here)
lgbm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LGBMRegressor(n_estimators=500, learning_rate=0.1, num_leaves=50, random_state=42, verbose=-1))
])

lgbm_pipeline.fit(X_train, y_train)
y_pred = lgbm_pipeline.predict(X_test)

### 5. Evaluation

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R2 Score: {r2:.4f}")

In [ ]:
# Actual vs Predicted Plot
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Actual vs Predicted House Values')
plt.show()

In [ ]:
# Feature Importance
importances = lgbm_pipeline.named_steps['regressor'].feature_importances_
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(10, 6))
sns.barplot(x=importances[indices], y=[housing.feature_names[i] for i in indices])
plt.title('Top Features Determining House Price')
plt.show()